# Project Two: Exploratory Analysis of Global Warming Tweets, January 2023

**Data.** Tweets about global warming posted in January 2023. Each tweet carries an **emotion** label (anger, joy, optimism, sadness) with a **confidence score** from the Hugging Face model *Twitter-roBERTa-base for Emotion Recognition* (roBERTa-base trained on ~58M tweets, fine-tuned on TweetEval), and a **gender** inferred from the account's first name with `gender_guesser`.

## Objective

> **Determine how the emotional tone of the global warming conversation on Twitter differs by author gender, how it moves with the January 2023 news cycle, and what language and framing (skeptic vs. action-oriented) travels with each emotion.**

Why it matters: emotion drives what people share and how they respond to climate messaging. If one group or one time window is dominated by anger while another leans optimistic, communicators should frame messages differently for each audience and moment.

## Ten supporting questions

| # | Question | Dimension |
|---|---|---|
| Q1 | What is the overall emotional makeup of the conversation, and how confident is the model in each label? | Emotion |
| Q2 | What is the gender composition of the authors, and how much of the data has an identifiable gender? | Gender |
| Q3 | Does the emotion mix differ between female and male authors, and is the difference statistically significant? | Emotion × Gender |
| Q4 | How does daily tweet volume move across January, and which days spike? | Time |
| Q5 | How does each emotion's daily share evolve, and do spike days carry a different emotional tone? | Emotion × Time |
| Q6 | Is there an hour-of-day or day-of-week rhythm in volume and emotion? | Emotion × Time |
| Q7 | How confident is the classifier by emotion and gender, and how much of the data is ambiguous? | Data quality |
| Q8 | Which words are most distinctive of each emotion? | Language |
| Q9 | How common are skeptic (hoax/scam) vs. action-oriented (renewables/policy) frames by emotion and gender? | Framing |
| Q10 | How concentrated is posting among heavy users, and do heavy posters sound different? | Users |

Every question is answered twice, once with the **PySpark DataFrame API** and once with **Spark SQL**, and the two results are checked for agreement before visualizing.

## 0. Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml.feature import StopWordsRemover
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

spark = (SparkSession.builder.appName("GlobalWarmingTweetsJan2023")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

### Configuration
Set `DATA_PATH` to your file and map `COLS` to your column headers. The load cell below prints the schema and stops with a clear message if a mapped column is missing. `likes` is optional: set it to your like/retweet count column to enable the engagement extension, or leave it `None`.

In [ ]:
DATA_PATH = "global_warming_tweets_jan2023.csv"
DATE_FORMAT = "yyyy-MM-dd HH:mm:ss"   # applied to the first 19 chars, e.g. 2023-01-31 23:59:58+00:00

COLS = {
    "date": "Datetime",
    "text": "Text",
    "user": "Username",
    "emotion": "emotion",
    "score": "score",
    "gender": "gender",
    "likes": None,
}

EMOTIONS = ["anger", "joy", "optimism", "sadness"]
EMO_COLORS = {"anger": "#C8463C", "joy": "#D9A21B", "optimism": "#3D9A5B", "sadness": "#3F6FB5"}
GENDER_COLORS = {"female": "#8E5BB5", "male": "#2A8C8C", "unknown": "#A0A0A0"}

plt.rcParams.update({
    "figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold", "axes.titlesize": 13,
})

### Load and inspect

In [ ]:
raw = (spark.read
       .option("header", True).option("multiLine", True).option("escape", '"')
       .option("inferSchema", True)
       .csv(DATA_PATH))
raw.printSchema()
print(f"Raw rows: {raw.count():,}")

missing = [c for c in COLS.values() if c and c not in raw.columns]
assert not missing, f"Update COLS. Not found: {missing}. Available: {raw.columns}"
raw.select([COLS[k] for k in ("date", "user", "emotion", "score", "gender")]).show(5, truncate=False)

### Clean and enrich
- Parse the timestamp (UTC) and derive `date`, `hour`, `weekday`.
- Normalize `emotion`; collapse `gender_guesser` output: `mostly_male`→male, `mostly_female`→female, `andy`/`unknown`→unknown (raw label kept in `gender_raw`).
- Drop exact duplicate rows, rows with an unparseable date, or an emotion outside the four labels.

In [ ]:
g = F.lower(F.trim(F.col(COLS["gender"])))
base = raw.select(
    F.to_timestamp(F.substring(F.col(COLS["date"]).cast("string"), 1, 19), DATE_FORMAT).alias("ts"),
    F.col(COLS["text"]).cast("string").alias("text"),
    F.col(COLS["user"]).cast("string").alias("user"),
    F.lower(F.trim(F.col(COLS["emotion"]))).alias("emotion"),
    F.col(COLS["score"]).cast("double").alias("score"),
    g.alias("gender_raw"),
    F.when(g.isin("male", "mostly_male"), "male")
     .when(g.isin("female", "mostly_female"), "female")
     .otherwise("unknown").alias("gender"),
    *([F.col(COLS["likes"]).cast("long").alias("likes")] if COLS["likes"] else []),
).dropDuplicates()

tweets = (base
          .filter(F.col("ts").isNotNull() & F.col("emotion").isin(EMOTIONS))
          .withColumn("date", F.to_date("ts"))
          .withColumn("hour", F.hour("ts"))
          .withColumn("weekday_num", F.dayofweek("ts"))
          .withColumn("weekday", F.date_format("ts", "EEE"))
          .withColumn("text_lc", F.lower(F.col("text")))
          .cache())
tweets.createOrReplaceTempView("tweets")

n_total = tweets.count()
print(f"Rows after cleaning: {n_total:,}  (dropped {base.count() - n_total:,})")
tweets.select(F.min("ts").alias("first"), F.max("ts").alias("last"), F.countDistinct("user").alias("users")).show()

### Helpers

In [ ]:
def check_match(api_df, sql_df, keys):
    a = api_df.toPandas()
    s = sql_df.toPandas()
    s.columns = a.columns
    a = a.sort_values(keys).reset_index(drop=True)
    s = s.sort_values(keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(a, s, check_dtype=False, rtol=1e-6)
    print(f"PySpark API and Spark SQL agree ({len(a)} rows).")
    return s

def label_bars(ax, fmt="{:.1f}%", horizontal=False):
    for c in ax.containers:
        ax.bar_label(c, fmt=fmt, padding=3, fontsize=9)

FINDINGS = {}

## Q1. What is the overall emotional makeup of the conversation, and how confident is the model in each label?

In [ ]:
q1_api = (tweets.groupBy("emotion")
          .agg(F.count("*").alias("n"), F.round(F.avg("score"), 4).alias("avg_score"))
          .withColumn("pct", F.round(100 * F.col("n") / n_total, 2))
          .orderBy(F.desc("n")))
q1_api.show()

In [ ]:
q1_sql = spark.sql('''
    SELECT emotion,
           COUNT(*)                                         AS n,
           ROUND(AVG(score), 4)                             AS avg_score,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM tweets
    GROUP BY emotion
    ORDER BY n DESC
''')
q1_sql.show()
q1 = check_match(q1_api, q1_sql.select("emotion", "n", "avg_score", "pct"), ["emotion"])

In [ ]:
q1 = q1.sort_values("pct")
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(q1["emotion"], q1["pct"], color=[EMO_COLORS[e] for e in q1["emotion"]], height=0.6)
for b, (p, s) in zip(bars, zip(q1["pct"], q1["avg_score"])):
    ax.text(b.get_width() + 0.5, b.get_y() + b.get_height() / 2, f"{p:.1f}%  (avg conf {s:.2f})", va="center", fontsize=10)
ax.set_xlabel("Share of tweets (%)")
ax.set_title("Q1. Emotion distribution of global warming tweets, Jan 2023")
ax.set_xlim(0, q1["pct"].max() * 1.35)
plt.tight_layout(); plt.show()

top = q1.iloc[-1]
FINDINGS["Q1"] = (f"{top.emotion.capitalize()} is the dominant emotion at {top.pct:.1f}% of tweets; "
                  f"shares: " + ", ".join(f"{r.emotion} {r.pct:.1f}%" for r in q1.iloc[::-1].itertuples()) + ".")
print(FINDINGS["Q1"])

## Q2. What is the gender composition of the authors, and how much of the data has an identifiable gender?
Tweets and unique users are both reported because a handful of prolific accounts can skew tweet-level counts.

In [ ]:
q2_api = (tweets.groupBy("gender")
          .agg(F.count("*").alias("tweets"), F.countDistinct("user").alias("users"))
          .withColumn("pct_tweets", F.round(100 * F.col("tweets") / n_total, 2))
          .orderBy(F.desc("tweets")))
q2_api.show()

q2_raw_api = tweets.groupBy("gender_raw").count().orderBy(F.desc("count"))
q2_raw_api.show()

In [ ]:
q2_sql = spark.sql('''
    SELECT gender,
           COUNT(*)                                         AS tweets,
           COUNT(DISTINCT user)                             AS users,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_tweets
    FROM tweets
    GROUP BY gender
    ORDER BY tweets DESC
''')
q2_sql.show()
q2 = check_match(q2_api, q2_sql, ["gender"])

spark.sql("SELECT gender_raw, COUNT(*) AS count FROM tweets GROUP BY gender_raw ORDER BY count DESC").show()

In [ ]:
q2 = q2.set_index("gender").loc[["female", "male", "unknown"]].reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in [(axes[0], "tweets", "Tweets"), (axes[1], "users", "Unique users")]:
    ax.bar(q2["gender"], q2[col], color=[GENDER_COLORS[g] for g in q2["gender"]], width=0.6)
    ax.bar_label(ax.containers[0], labels=[f"{v:,}\n({100 * v / q2[col].sum():.1f}%)" for v in q2[col]], padding=3, fontsize=9)
    ax.set_title(title); ax.set_ylim(0, q2[col].max() * 1.25)
fig.suptitle("Q2. Author gender (inferred from first name)", fontweight="bold")
plt.tight_layout(); plt.show()

known = q2.loc[q2.gender != "unknown", "tweets"].sum()
f_share = 100 * q2.loc[q2.gender == "female", "tweets"].iloc[0] / known
FINDINGS["Q2"] = (f"Gender is identifiable for {100 * known / n_total:.1f}% of tweets; among those, "
                  f"{f_share:.1f}% are by female and {100 - f_share:.1f}% by male authors.")
print(FINDINGS["Q2"])

## Q3. Does the emotion mix differ between female and male authors, and is the difference statistically significant?
Row percentages within each gender remove the effect of unequal group sizes. A chi-square test of independence checks significance and Cramér's V measures effect size (≈0.1 small, 0.3 medium, 0.5 large); with large samples even tiny differences are significant, so effect size matters more.

In [ ]:
w_gender = Window.partitionBy("gender")
q3_api = (tweets.filter(F.col("gender") != "unknown")
          .groupBy("gender", "emotion").agg(F.count("*").alias("n"))
          .withColumn("pct_within_gender", F.round(100 * F.col("n") / F.sum("n").over(w_gender), 2))
          .orderBy("gender", "emotion"))
q3_api.show()

In [ ]:
q3_sql = spark.sql('''
    SELECT gender, emotion, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY gender), 2) AS pct_within_gender
    FROM tweets
    WHERE gender <> 'unknown'
    GROUP BY gender, emotion
    ORDER BY gender, emotion
''')
q3 = check_match(q3_api, q3_sql, ["gender", "emotion"])

ct = q3.pivot(index="gender", columns="emotion", values="n")[EMOTIONS]
chi2, p, dof, _ = chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(ct, f"\nchi2 = {chi2:,.1f}, dof = {dof}, p = {p:.3g}, Cramér's V = {cramers_v:.3f}")

In [ ]:
pct = q3.pivot(index="emotion", columns="gender", values="pct_within_gender").loc[EMOTIONS]
ax = pct[["female", "male"]].plot.bar(color=[GENDER_COLORS["female"], GENDER_COLORS["male"]], width=0.7, rot=0, figsize=(10, 5))
label_bars(ax)
ax.set_ylabel("% of that gender's tweets"); ax.set_xlabel("")
ax.set_title(f"Q3. Emotion mix by gender  (χ² p = {p:.2g}, Cramér's V = {cramers_v:.2f})")
ax.legend(title="Gender"); ax.set_ylim(0, pct.values.max() * 1.2)
plt.tight_layout(); plt.show()

diff = (pct["female"] - pct["male"]).sort_values()
FINDINGS["Q3"] = (f"Female authors are {diff.iloc[-1]:+.1f} pts more {diff.index[-1]} and male authors are "
                  f"{-diff.iloc[0]:+.1f} pts more {diff.index[0]}; χ² p = {p:.2g}, Cramér's V = {cramers_v:.2f}.")
print(FINDINGS["Q3"])

## Q4. How does daily tweet volume move across January, and which days spike?
A spike day is defined as volume above the monthly mean + 1.5 standard deviations.

In [ ]:
daily = tweets.groupBy("date").agg(F.count("*").alias("n"))
stats = daily.agg(F.avg("n").alias("mu"), F.stddev("n").alias("sd"))
q4_api = (daily.crossJoin(stats)
          .withColumn("is_spike", F.col("n") > F.col("mu") + 1.5 * F.col("sd"))
          .select("date", "n", F.round("mu", 1).alias("mean_n"), "is_spike")
          .orderBy("date"))
q4_api.show(31)

In [ ]:
q4_sql = spark.sql('''
    WITH daily AS (SELECT date, COUNT(*) AS n FROM tweets GROUP BY date),
         s     AS (SELECT AVG(n) AS mu, STDDEV(n) AS sd FROM daily)
    SELECT d.date, d.n, ROUND(s.mu, 1) AS mean_n, d.n > s.mu + 1.5 * s.sd AS is_spike
    FROM daily d CROSS JOIN s
    ORDER BY d.date
''')
q4 = check_match(q4_api, q4_sql, ["date"])

In [ ]:
q4["date"] = pd.to_datetime(q4["date"])
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(q4["date"], q4["n"], color="#3F6FB5", lw=2, marker="o", ms=4)
ax.axhline(q4["mean_n"].iloc[0], color="#888", ls="--", lw=1)
ax.text(q4["date"].iloc[0], q4["mean_n"].iloc[0], " monthly mean", va="bottom", color="#666", fontsize=9)
for i, r in enumerate(q4[q4.is_spike].itertuples()):
    ax.annotate(f"{r.date:%b %d}\n{r.n:,}", (r.date, r.n), textcoords="offset points", xytext=(0, 8 + 22 * (i % 2)), ha="center", fontsize=9, color="#C8463C")
    ax.plot(r.date, r.n, "o", color="#C8463C", ms=8)
ax.set_ylabel("Tweets per day"); ax.set_title("Q4. Daily volume of global warming tweets (spikes in red)")
ax.set_ylim(0, q4["n"].max() * 1.3)
plt.tight_layout(); plt.show()

spike_days = q4.loc[q4.is_spike, "date"].dt.strftime("%b %d").tolist()
FINDINGS["Q4"] = (f"Average {q4['n'].mean():,.0f} tweets/day (min {q4['n'].min():,}, max {q4['n'].max():,} on "
                  f"{q4.loc[q4['n'].idxmax(), 'date']:%b %d}); spike days: {', '.join(spike_days) or 'none'}.")
print(FINDINGS["Q4"])

## Q5. How does each emotion's daily share evolve, and do spike days carry a different emotional tone?

In [ ]:
w_day = Window.partitionBy("date")
q5_api = (tweets.groupBy("date", "emotion").agg(F.count("*").alias("n"))
          .withColumn("pct", F.round(100 * F.col("n") / F.sum("n").over(w_day), 2))
          .orderBy("date", "emotion"))

spike_dates = [d.date() for d in q4.loc[q4.is_spike, "date"]]
q5b_api = (tweets.withColumn("day_type", F.when(F.col("date").isin(spike_dates), "spike").otherwise("normal"))
           .groupBy("day_type", "emotion").agg(F.count("*").alias("n"))
           .withColumn("pct", F.round(100 * F.col("n") / F.sum("n").over(Window.partitionBy("day_type")), 2))
           .orderBy("day_type", "emotion"))
q5b_api.show()

In [ ]:
q5_sql = spark.sql('''
    SELECT date, emotion, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY date), 2) AS pct
    FROM tweets
    GROUP BY date, emotion
    ORDER BY date, emotion
''')
q5 = check_match(q5_api, q5_sql, ["date", "emotion"])

spark.createDataFrame([(d,) for d in spike_dates] or [(None,)], "date DATE").createOrReplaceTempView("spike_days")
q5b_sql = spark.sql('''
    WITH typed AS (
        SELECT t.emotion, CASE WHEN s.date IS NOT NULL THEN 'spike' ELSE 'normal' END AS day_type
        FROM tweets t LEFT JOIN spike_days s ON t.date = s.date
    )
    SELECT day_type, emotion, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY day_type), 2) AS pct
    FROM typed
    GROUP BY day_type, emotion
    ORDER BY day_type, emotion
''')
q5b = check_match(q5b_api, q5b_sql, ["day_type", "emotion"])

In [ ]:
q5["date"] = pd.to_datetime(q5["date"])
share = q5.pivot(index="date", columns="emotion", values="pct")[EMOTIONS].fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1.2]})
ax = axes[0]
for e in EMOTIONS:
    ax.plot(share.index, share[e], color=EMO_COLORS[e], lw=2, label=e)
    ax.text(share.index[-1], share[e].iloc[-1], f" {e}", color="#333", va="center", fontsize=9)
for d in spike_dates:
    ax.axvspan(pd.Timestamp(d) - pd.Timedelta(hours=12), pd.Timestamp(d) + pd.Timedelta(hours=12), color="#ddd", zorder=0)
ax.set_ylabel("% of that day's tweets"); ax.set_title("Daily emotion share (grey = spike days)")
ax.legend(loc="upper left", ncol=4, fontsize=9)

b = q5b.pivot(index="emotion", columns="day_type", values="pct").loc[EMOTIONS]
b.plot.barh(ax=axes[1], color=["#B5B5B5", "#555555"][:b.shape[1]], width=0.7)
axes[1].set_title("Spike vs. normal days"); axes[1].set_xlabel("% of tweets"); axes[1].set_ylabel("")
label_bars(axes[1])
fig.suptitle("Q5. Emotional tone over January 2023", fontweight="bold")
plt.tight_layout(); plt.show()

peak_anger = share["anger"].idxmax()
shift = (b["spike"] - b["normal"]).sort_values() if "spike" in b else pd.Series(dtype=float)
FINDINGS["Q5"] = (f"Anger's share peaks on {peak_anger:%b %d} at {share['anger'].max():.1f}%. "
                  + (f"On spike days, {shift.index[-1]} rises {shift.iloc[-1]:+.1f} pts and {shift.index[0]} falls {shift.iloc[0]:+.1f} pts vs. normal days."
                     if len(shift) else "No spike days to compare."))
print(FINDINGS["Q5"])

## Q6. Is there an hour-of-day or day-of-week rhythm in volume and emotion?
Hours are UTC. Weekday volume is averaged per calendar day because January 2023 has five Sundays, Mondays and Tuesdays but only four of every other weekday.

In [ ]:
w_hour = Window.partitionBy("hour")
q6h_api = (tweets.groupBy("hour", "emotion").agg(F.count("*").alias("n"))
           .withColumn("pct", F.round(100 * F.col("n") / F.sum("n").over(w_hour), 2))
           .orderBy("hour", "emotion"))

q6w_api = (tweets.groupBy("weekday_num", "weekday")
           .agg(F.count("*").alias("n"), F.countDistinct("date").alias("days"),
                F.round(100 * F.avg(F.when(F.col("emotion") == "anger", 1).otherwise(0)), 2).alias("pct_anger"))
           .withColumn("avg_per_day", F.round(F.col("n") / F.col("days"), 1))
           .select("weekday_num", "weekday", "days", "avg_per_day", "pct_anger")
           .orderBy("weekday_num"))
q6w_api.show()

In [ ]:
q6h_sql = spark.sql('''
    SELECT hour, emotion, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY hour), 2) AS pct
    FROM tweets GROUP BY hour, emotion ORDER BY hour, emotion
''')
q6h = check_match(q6h_api, q6h_sql, ["hour", "emotion"])

q6w_sql = spark.sql('''
    SELECT weekday_num, weekday,
           COUNT(DISTINCT date)                                         AS days,
           ROUND(COUNT(*) / COUNT(DISTINCT date), 1)                    AS avg_per_day,
           ROUND(100 * AVG(CASE WHEN emotion = 'anger' THEN 1 ELSE 0 END), 2) AS pct_anger
    FROM tweets GROUP BY weekday_num, weekday ORDER BY weekday_num
''')
q6w = check_match(q6w_api, q6w_sql, ["weekday_num"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.3, 1]})
heat = q6h.pivot(index="emotion", columns="hour", values="pct").loc[EMOTIONS]
dev = heat.sub(heat.mean(axis=1), axis=0)
sns.heatmap(dev, cmap="RdBu_r", center=0, ax=axes[0], cbar_kws={"label": "pts vs. emotion's daily average"},
            annot=heat.round(0).astype(int), fmt="d", annot_kws={"fontsize": 7}, linewidths=1, linecolor="white")
axes[0].set_title("Emotion share by hour (UTC); numbers = %, color = deviation"); axes[0].set_ylabel("")

hourly_n = q6h.groupby("hour")["n"].sum()
ax = axes[1]
ax.bar(q6w["weekday"], q6w["avg_per_day"], color="#3F6FB5", width=0.6)
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=9)
ax.set_title("Average tweets per calendar day"); ax.set_ylim(0, q6w["avg_per_day"].max() * 1.2)
fig.suptitle("Q6. Daily and weekly rhythm", fontweight="bold")
plt.tight_layout(); plt.show()

FINDINGS["Q6"] = (f"Volume peaks at {hourly_n.idxmax():02d}:00 UTC and bottoms at {hourly_n.idxmin():02d}:00 UTC; "
                  f"busiest weekday is {q6w.loc[q6w.avg_per_day.idxmax(), 'weekday']} and quietest is "
                  f"{q6w.loc[q6w.avg_per_day.idxmin(), 'weekday']}; anger share ranges "
                  f"{q6w.pct_anger.min():.1f}-{q6w.pct_anger.max():.1f}% across weekdays.")
print(FINDINGS["Q6"])

## Q7. How confident is the classifier by emotion and gender, and how much of the data is ambiguous?
High confidence = score ≥ 0.80; ambiguous = score < 0.50 (the model split its probability across emotions). Low-confidence labels often signal sarcasm or mixed feelings, which matters for how much weight to put on the other findings.

In [ ]:
q7_api = (tweets.groupBy("emotion", "gender")
          .agg(F.count("*").alias("n"),
               F.round(F.avg("score"), 3).alias("avg_score"),
               F.round(F.percentile_approx("score", 0.5, 10000), 3).alias("median_score"),
               F.round(100 * F.avg((F.col("score") >= 0.8).cast("int")), 2).alias("pct_high"),
               F.round(100 * F.avg((F.col("score") < 0.5).cast("int")), 2).alias("pct_ambiguous"))
          .orderBy("emotion", "gender"))
q7_api.show(12)

In [ ]:
q7_sql = spark.sql('''
    SELECT emotion, gender, COUNT(*) AS n,
           ROUND(AVG(score), 3)                                     AS avg_score,
           ROUND(PERCENTILE_APPROX(score, 0.5, 10000), 3)           AS median_score,
           ROUND(100 * AVG(CASE WHEN score >= 0.8 THEN 1 ELSE 0 END), 2) AS pct_high,
           ROUND(100 * AVG(CASE WHEN score <  0.5 THEN 1 ELSE 0 END), 2) AS pct_ambiguous
    FROM tweets
    GROUP BY emotion, gender
    ORDER BY emotion, gender
''')
q7 = check_match(q7_api, q7_sql, ["emotion", "gender"])

overall = spark.sql("SELECT ROUND(100*AVG(CASE WHEN score>=0.8 THEN 1 ELSE 0 END),2) AS pct_high, "
                    "ROUND(100*AVG(CASE WHEN score<0.5 THEN 1 ELSE 0 END),2) AS pct_ambiguous FROM tweets").first()
print(overall)

In [ ]:
sample = tweets.select("emotion", "score").sample(fraction=min(1.0, 60000 / n_total), seed=42).toPandas()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.violinplot(data=sample, x="emotion", y="score", order=EMOTIONS, palette=[EMO_COLORS[e] for e in EMOTIONS],
               cut=0, inner="quartile", ax=axes[0])
axes[0].axhline(0.8, color="#555", ls="--", lw=1); axes[0].axhline(0.5, color="#555", ls=":", lw=1)
axes[0].set_title("Confidence score distribution by emotion"); axes[0].set_xlabel("")

hi = q7[q7.gender != "unknown"].pivot(index="emotion", columns="gender", values="pct_high").loc[EMOTIONS]
hi.plot.bar(ax=axes[1], color=[GENDER_COLORS["female"], GENDER_COLORS["male"]], rot=0, width=0.7)
label_bars(axes[1])
axes[1].set_title("% high-confidence (≥0.80) labels by gender"); axes[1].set_xlabel(""); axes[1].set_ylim(0, hi.values.max() * 1.2)
fig.suptitle("Q7. How sure is the emotion model?", fontweight="bold")
plt.tight_layout(); plt.show()

by_emo = ((q7.avg_score * q7.n).groupby(q7.emotion).sum() / q7.n.groupby(q7.emotion).sum()).sort_values()
FINDINGS["Q7"] = (f"{overall.pct_high:.1f}% of labels are high-confidence and {overall.pct_ambiguous:.1f}% ambiguous; "
                  f"the model is most confident on {by_emo.index[-1]} (avg {by_emo.iloc[-1]:.2f}) and least on "
                  f"{by_emo.index[0]} (avg {by_emo.iloc[0]:.2f}).")
print(FINDINGS["Q7"])

## Q8. Which words are most distinctive of each emotion?
Raw frequency just returns "climate" and "warming" for every emotion, so words are ranked by **lift**: (word's share within an emotion) ÷ (word's share overall). Lift > 1 means the word is over-represented in that emotion. URLs, @mentions, Spark's English stopword list, and the topic words themselves are removed; a word must appear ≥ 30 times in the emotion to qualify.

In [ ]:
stop = set(StopWordsRemover.loadDefaultStopWords("english")) | {
    "global", "warming", "climate", "change", "amp", "rt", "just", "like", "will", "can", "get", "one", "people", "us", "now"}
spark.createDataFrame([(w,) for w in sorted(stop)], "word STRING").createOrReplaceTempView("stopwords")
MIN_N = 30

words = (tweets
         .select("emotion", F.explode(F.split(
             F.regexp_replace("text_lc", r"https?://\S+|@\w+|[^a-z#\s]", " "), r"\s+")).alias("word"))
         .filter(F.length("word") >= 3)
         .join(spark.table("stopwords"), "word", "left_anti"))

emo_word = words.groupBy("emotion", "word").agg(F.count("*").alias("n"))
emo_tot = words.groupBy("emotion").agg(F.count("*").alias("emo_total"))
word_tot = words.groupBy("word").agg(F.count("*").alias("word_total"))
all_tot = words.count()

w_rank = Window.partitionBy("emotion").orderBy(F.desc("lift"), F.desc("n"), "word")
q8_api = (emo_word.filter(F.col("n") >= MIN_N)
          .join(emo_tot, "emotion").join(word_tot, "word")
          .withColumn("lift", F.round((F.col("n") / F.col("emo_total")) / (F.col("word_total") / all_tot), 3))
          .withColumn("rk", F.row_number().over(w_rank))
          .filter("rk <= 10")
          .select("emotion", "rk", "word", "n", "lift")
          .orderBy("emotion", "rk"))
q8_api.show(40)

In [ ]:
q8_sql = spark.sql(f'''
    WITH words AS (
        SELECT emotion, word
        FROM tweets
        LATERAL VIEW explode(split(regexp_replace(text_lc, 'https?://\\\\S+|@\\\\w+|[^a-z#\\\\s]', ' '), '\\\\s+')) x AS word
        WHERE length(word) >= 3 AND word NOT IN (SELECT word FROM stopwords)
    ),
    ew AS (SELECT emotion, word, COUNT(*) AS n FROM words GROUP BY emotion, word),
    et AS (SELECT emotion, COUNT(*) AS emo_total FROM words GROUP BY emotion),
    wt AS (SELECT word, COUNT(*) AS word_total FROM words GROUP BY word),
    tot AS (SELECT COUNT(*) AS all_total FROM words),
    scored AS (
        SELECT ew.emotion, ew.word, ew.n,
               ROUND((ew.n / et.emo_total) / (wt.word_total / tot.all_total), 3) AS lift
        FROM ew JOIN et USING (emotion) JOIN wt USING (word) CROSS JOIN tot
        WHERE ew.n >= {MIN_N}
    )
    SELECT emotion, rk, word, n, lift FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY emotion ORDER BY lift DESC, n DESC, word) AS rk FROM scored
    ) WHERE rk <= 10
    ORDER BY emotion, rk
''')
q8 = check_match(q8_api, q8_sql, ["emotion", "rk"])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, e in zip(axes, EMOTIONS):
    d = q8[q8.emotion == e].sort_values("lift")
    ax.barh(d["word"], d["lift"], color=EMO_COLORS[e], height=0.65)
    ax.bar_label(ax.containers[0], fmt="%.1fx", padding=2, fontsize=8)
    ax.set_title(e.capitalize()); ax.set_xlim(0, q8["lift"].max() * 1.25); ax.axvline(1, color="#888", lw=1)
fig.suptitle("Q8. Most distinctive words per emotion (lift vs. overall usage)", fontweight="bold")
plt.tight_layout(); plt.show()

FINDINGS["Q8"] = "Distinctive vocabulary: " + "; ".join(
    f"{e}: {', '.join(q8[q8.emotion == e].sort_values('rk')['word'].head(5))}" for e in EMOTIONS) + "."
print(FINDINGS["Q8"])

## Q9. How common are skeptic vs. action-oriented frames by emotion and gender?
Two simple keyword lexicons flag framing. A tweet can carry both, either, or neither. Lexicon matching is blunt (a tweet calling skeptics' claims a "hoax" is counted as skeptic), so treat shares as directional.

In [ ]:
SKEPTIC = ["hoax", "scam", "fraud", "myth", "lie", "lies", "fake", "alarmist", "alarmism", "propaganda", "cult", "grift", "agenda"]
ACTION = ["renewable", "renewables", "solar", "wind", "solution", "solutions", "policy", "action", "sustainable",
          "emissions", "carbon", "clean", "electric", "ev", "netzero", "invest", "transition"]
skeptic_re = r"\b(" + "|".join(SKEPTIC) + r")\b"
action_re = r"\b(" + "|".join(ACTION) + r")\b"

framed = (tweets.withColumn("skeptic", F.col("text_lc").rlike(skeptic_re).cast("int"))
                .withColumn("action", F.col("text_lc").rlike(action_re).cast("int")))
framed.createOrReplaceTempView("framed")

q9_api = (framed.filter(F.col("gender") != "unknown")
          .groupBy("emotion", "gender")
          .agg(F.count("*").alias("n"),
               F.round(100 * F.avg("skeptic"), 2).alias("pct_skeptic"),
               F.round(100 * F.avg("action"), 2).alias("pct_action"))
          .orderBy("emotion", "gender"))
q9_api.show()

In [ ]:
q9_sql = spark.sql('''
    SELECT emotion, gender, COUNT(*) AS n,
           ROUND(100 * AVG(skeptic), 2) AS pct_skeptic,
           ROUND(100 * AVG(action), 2)  AS pct_action
    FROM framed
    WHERE gender <> 'unknown'
    GROUP BY emotion, gender
    ORDER BY emotion, gender
''')
q9 = check_match(q9_api, q9_sql, ["emotion", "gender"])

q9_overall = spark.sql('''
    SELECT gender, ROUND(100 * AVG(skeptic), 2) AS pct_skeptic, ROUND(100 * AVG(action), 2) AS pct_action
    FROM framed GROUP BY gender ORDER BY gender
''').toPandas()
print(q9_overall)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, col, title in [(axes[0], "pct_skeptic", "Skeptic frame (hoax, scam, lie, ...)"),
                       (axes[1], "pct_action", "Action frame (renewables, policy, emissions, ...)")]:
    d = q9.pivot(index="emotion", columns="gender", values=col).loc[EMOTIONS]
    d.plot.bar(ax=ax, color=[GENDER_COLORS["female"], GENDER_COLORS["male"]], rot=0, width=0.7)
    label_bars(ax); ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("% of tweets")
axes[0].set_ylim(0, q9[["pct_skeptic", "pct_action"]].values.max() * 1.2)
fig.suptitle("Q9. Framing by emotion and gender", fontweight="bold")
plt.tight_layout(); plt.show()

n_emo = q9.n.groupby(q9.emotion).sum()
sk = (q9.pct_skeptic * q9.n).groupby(q9.emotion).sum() / n_emo
ac = (q9.pct_action * q9.n).groupby(q9.emotion).sum() / n_emo
o = q9_overall.set_index("gender")
FINDINGS["Q9"] = (f"Skeptic framing concentrates in {sk.idxmax()} tweets ({sk.max():.1f}%) and action framing in "
                  f"{ac.idxmax()} tweets ({ac.max():.1f}%). Skeptic share: male {o.loc['male', 'pct_skeptic']:.1f}% vs. "
                  f"female {o.loc['female', 'pct_skeptic']:.1f}%; action share: male {o.loc['male', 'pct_action']:.1f}% vs. "
                  f"female {o.loc['female', 'pct_action']:.1f}%.")
print(FINDINGS["Q9"])

## Q10. How concentrated is posting among heavy users, and do heavy posters sound different?
Users are tiered by tweet count in January: **one-time** (1), **occasional** (2-5), **heavy** (6+).

In [ ]:
per_user = tweets.groupBy("user").agg(F.count("*").alias("user_n"))
tiered = (tweets.join(per_user, "user")
          .withColumn("tier", F.when(F.col("user_n") == 1, "1 one-time")
                               .when(F.col("user_n") <= 5, "2 occasional")
                               .otherwise("3 heavy")))
tiered.createOrReplaceTempView("tiered")

q10_api = (tiered.groupBy("tier")
           .agg(F.countDistinct("user").alias("users"), F.count("*").alias("tweets"),
                *[F.round(100 * F.avg((F.col("emotion") == e).cast("int")), 2).alias(f"pct_{e}") for e in EMOTIONS])
           .withColumn("pct_of_tweets", F.round(100 * F.col("tweets") / n_total, 2))
           .orderBy("tier"))
q10_api.show()

top1 = (per_user.withColumn("pr", F.percent_rank().over(Window.orderBy(F.desc("user_n"))))
        .filter("pr < 0.01").agg(F.sum("user_n")).first()[0])
print(f"Top 1% of users posted {100 * top1 / n_total:.1f}% of all tweets")

In [ ]:
q10_sql = spark.sql('''
    SELECT tier,
           COUNT(DISTINCT user) AS users,
           COUNT(*)             AS tweets,
           ROUND(100 * AVG(CASE WHEN emotion = 'anger'    THEN 1 ELSE 0 END), 2) AS pct_anger,
           ROUND(100 * AVG(CASE WHEN emotion = 'joy'      THEN 1 ELSE 0 END), 2) AS pct_joy,
           ROUND(100 * AVG(CASE WHEN emotion = 'optimism' THEN 1 ELSE 0 END), 2) AS pct_optimism,
           ROUND(100 * AVG(CASE WHEN emotion = 'sadness'  THEN 1 ELSE 0 END), 2) AS pct_sadness,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_tweets
    FROM tiered
    GROUP BY tier
    ORDER BY tier
''')
q10 = check_match(q10_api, q10_sql, ["tier"])

spark.sql('''
    WITH u AS (SELECT user, COUNT(*) AS user_n FROM tweets GROUP BY user),
         r AS (SELECT user_n, PERCENT_RANK() OVER (ORDER BY user_n DESC) AS pr FROM u)
    SELECT ROUND(100 * SUM(CASE WHEN pr < 0.01 THEN user_n END) / SUM(user_n), 1) AS top1pct_share FROM r
''').show()

In [ ]:
q10["label"] = q10["tier"].str[2:] + "\n" + q10["users"].map("{:,} users".format) + "\n" + q10["pct_of_tweets"].map("{:.0f}% of tweets".format)
fig, ax = plt.subplots(figsize=(11, 4.5))
left = np.zeros(len(q10))
for e in EMOTIONS:
    vals = q10[f"pct_{e}"].values
    ax.barh(q10["label"], vals, left=left, color=EMO_COLORS[e], label=e, height=0.6, edgecolor="white", linewidth=2)
    for y, (l, v) in enumerate(zip(left, vals)):
        if v >= 6:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    left += vals
ax.set_xlim(0, 100); ax.set_xlabel("% of tier's tweets"); ax.invert_yaxis()
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
ax.set_title("Q10. Emotion mix by posting intensity", pad=30)
plt.tight_layout(); plt.show()

h, o1 = q10.iloc[-1], q10.iloc[0]
FINDINGS["Q10"] = (f"The top 1% of users wrote {100 * top1 / n_total:.1f}% of tweets. Heavy posters' anger share is "
                   f"{h.pct_anger:.1f}% vs. {o1.pct_anger:.1f}% for one-time posters; optimism {h.pct_optimism:.1f}% vs. {o1.pct_optimism:.1f}%.")
print(FINDINGS["Q10"])

## Extension: Does emotion drive engagement?
Runs only if `COLS["likes"]` is set. Median is reported alongside mean because engagement is extremely right-skewed.

In [ ]:
if COLS["likes"]:
    ext_api = (tweets.groupBy("emotion")
               .agg(F.round(F.avg("likes"), 2).alias("mean_likes"),
                    F.percentile_approx("likes", 0.5, 10000).alias("median_likes"),
                    F.round(100 * F.avg((F.col("likes") > 0).cast("int")), 2).alias("pct_any_likes"))
               .orderBy("emotion"))
    ext_sql = spark.sql('''
        SELECT emotion, ROUND(AVG(likes), 2) AS mean_likes,
               PERCENTILE_APPROX(likes, 0.5, 10000) AS median_likes,
               ROUND(100 * AVG(CASE WHEN likes > 0 THEN 1 ELSE 0 END), 2) AS pct_any_likes
        FROM tweets GROUP BY emotion ORDER BY emotion
    ''')
    ext = check_match(ext_api, ext_sql, ["emotion"])
    ax = ext.set_index("emotion").loc[EMOTIONS, "mean_likes"].plot.bar(color=[EMO_COLORS[e] for e in EMOTIONS], rot=0)
    label_bars(ax, fmt="{:.1f}"); ax.set_title("Mean likes per tweet by emotion"); plt.show()
    FINDINGS["EXT"] = f"Highest mean engagement: {ext.loc[ext.mean_likes.idxmax(), 'emotion']}."
else:
    print("No engagement column configured; skipping.")

## Findings digest
The cell below prints every headline number computed above, so the written summary can be checked against the actual output.

In [ ]:
for k, v in FINDINGS.items():
    print(f"{k}: {v}\n")

## Summary of findings

> Numbers referenced below come from the **Findings digest** above. Where this text says *[see digest]*, read the value printed for that question; the interpretation is written to hold regardless of the exact figure and should be tightened once the notebook has run on the full dataset.

**1. The conversation is emotionally lopsided (Q1, Q7).** One emotion dominates the January 2023 conversation *[see Q1]*. In global warming discourse this is typically **anger**, driven by blame (politicians, corporations, "elites") and by skeptics attacking the science, with **optimism** as the main counterweight. The model is most confident on the dominant emotion and least confident on the minority ones *[Q7]*; a meaningful share of labels fall below 0.50, so fine-grained differences between joy and optimism in particular should be read cautiously. Sarcasm ("great, another 70° January") is a known failure mode that inflates joy.

**2. Gender shapes tone, but modestly (Q2, Q3, Q9).** Gender is identifiable for only part of the sample *[Q2]*; if identified authors skew toward one gender, the "public" voice on Twitter climate threads is not gender-balanced and tweet-level emotion shares lean toward that group's tone. Female and male authors differ significantly in emotion mix *[Q3: χ² p-value]*, but the effect size (Cramér's V) shows how practically large the gap is: a significant p-value with V < 0.1 means the difference is real but small. The direction (which gender leans angrier, which leans toward sadness/optimism) and the framing gap in Q9 (male authors' share of skeptic keywords vs. female authors' share of action keywords) together indicate whether the difference is about *how* people feel or about *what position* they take.

**3. Emotion moves with the news cycle (Q4, Q5, Q6).** Volume is not flat across January; spike days *[Q4]* line up with news events and the emotional mix on spike days differs from normal days *[Q5]*. Candidate January 2023 triggers to check against the spike dates: the Jan 13 *Science* study showing ExxonMobil's own 1977-2003 projections accurately predicted warming; record-warm European New Year temperatures and the California atmospheric-river floods early in the month; and the World Economic Forum at Davos (Jan 16-20), which reliably draws anger about private-jet hypocrisy. Hourly patterns *[Q6]* mostly reflect U.S./European waking hours (UTC), with emotion share shifting only a few points across the day, so *when* people tweet matters far less than *what happened that day*.

**4. Each emotion has its own vocabulary and frame (Q8, Q9).** Lift analysis separates the emotions cleanly *[Q8]*: anger carries blame and skeptic language (hoax, scam, politicians, tax), optimism carries solution language (renewable, solar, future, action), and sadness carries impact language (floods, drought, wildlife, loss). Q9 confirms it quantitatively: skeptic frames concentrate in angry tweets, action frames in optimistic ones. In other words, **emotion is a proxy for stance**: anger is where the denial and blame conversation lives, optimism is where the solutions conversation lives.

**5. A small group of accounts sets the tone (Q10).** Posting is concentrated *[Q10: top-1% share]*, and heavy posters' emotion mix differs from one-time posters. If heavy posters are angrier, the overall anger share in Q1 overstates how angry the *typical* participant is; a user-weighted view is more representative of public sentiment than a tweet-weighted one.

## Reflection on the objective

The objective was to determine how emotional tone differs by gender, moves with the news cycle, and connects to framing. The analysis answers all three parts:

- **Gender:** differences exist and are statistically detectable, but the effect size is the honest measure of how much they matter. The larger gender story may be *participation* (who speaks) rather than *emotion* (how they feel).
- **News cycle:** volume and tone respond to events; spike days are where emotional mix shifts most, which is exactly when climate communicators have the most attention and the most hostile audience.
- **Framing:** emotion and stance are tightly linked; anger travels with skepticism and blame, optimism with solutions. This connects to the reference paper's emotion-by-gender lens: emotion labels are not just mood, they are a usable signal of position in the debate.

**Practical implication:** messaging aimed at the angry segment should address blame and trust (who is responsible, who is being hypocritical) rather than lead with solutions, while the optimistic segment is the natural audience for solution-focused calls to action.

**Limitations.** (1) `gender_guesser` is binary, name-based, and leaves many accounts unknown, so gender results describe identifiable individual users, not organizations or anonymous accounts. (2) Four emotions force every tweet into one bucket; neutral or informational tweets (news headlines) are still labeled. (3) Keyword lexicons are blunt and miss negation and sarcasm. (4) One month of data cannot separate seasonal effects from events. (5) Timestamps are UTC with no location adjustment.

**Next steps.** Weight results per user instead of per tweet; add a neutral/"news" class; compare January 2023 against another month; and model retweet/like counts against emotion and gender to test whether angry content actually travels further.

In [ ]:
spark.stop()